# 05 — Elo rating

Adds an Elo rating as a feature: one number per fighter, corrected after every
fight, so it is the only feature that knows how good a fighter's opponents
were (`src/ufc_predictor/features/elo.py`).

Three things were tested before any of this code was written, and they shape
the notebook:

- **K is chosen by how much the rating helps the model**, not by how well Elo
  forecasts on its own. The K that forecasts best alone (≈63) added nothing to
  the model; a faster rating (K ≈ 150–300) helped in 8 of 8 model seeds.
- **Every comparison is averaged over several model seeds.** The model looks
  at a random 43% of features at each split, so the same settings score
  differently from seed to seed (spread ≈ 0.001). The first tuning run's
  0.6705 was the luckiest of 8 seeds.
- **`model_params.json` had been overwritten by a test** (fixed in
  `data/io.py`), so the settings are re-tuned here rather than trusted.

Order: tune without Elo → Elo on its own (reference) → search for K → check the
search's K with the model's own settings → rebuild the table with Elo → re-tune
with Elo → one look at validation. The test years stay closed. Runtime: about an
hour and a quarter, almost all of it the three tuning runs.

In [1]:
import numpy as np
import optuna
import pandas as pd

optuna.logging.set_verbosity(optuna.logging.WARNING)   # one line per trial would bury the results

from ufc_predictor.data.io import load_fighters, load_processed, save_processed
from ufc_predictor.features import manifest
from ufc_predictor.features.elo import ELO_COLUMNS, elo_forecast_log_loss, save_elo_params
from ufc_predictor.features.pipeline import build_featured_table
from ufc_predictor.features.shrinkage import load_alphas
from ufc_predictor.modeling.evaluate import compare, group_importance, is_womens_fight, score, scorecard
from ufc_predictor.modeling.model import fit_model, predict_proba
from ufc_predictor.modeling.split import chronological_split
from ufc_predictor.modeling.tune import choose_elo_k, cv_log_loss, save_params, tune, tuning_table

fights = load_processed("fights")
fighters = load_fighters()
alphas = load_alphas()

without_elo = build_featured_table(fights, fighters, alphas, elo_k=None)
train_without, validation_without, _ = chronological_split(without_elo)
print(f"{without_elo.shape[0]:,} fights x {without_elo.shape[1]} columns | training fights: {len(train_without):,}")

8,911 fights x 370 columns | training fights: 4,309


## 1. Tune the model without Elo

Gives the "without Elo" settings that the K search and the final comparison
need, and restores `model_params.json` honestly. Same search as before (60
Optuna trials, seeded), but each trial is now averaged over 3 model seeds so
the search can't reward a lucky draw.

In [2]:
study_without = tune(train_without, n_trials=60)
save_params(study_without)
params_without = study_without.best_params
print(f"best CV log loss, averaged over 3 seeds: {study_without.best_value:.5f}")
tuning_table(study_without, top=5)

best CV log loss, averaged over 3 seeds: 0.66097


,trial,log_loss,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features
0,46,0.66097,0.01619,550,15,8,2.29850,0.68860
1,53,0.66119,0.01325,700,18,6,3.17521,0.63208
2,55,0.66138,0.01312,700,20,6,2.29220,0.72258
3,59,0.66145,0.01194,800,18,5,1.91275,0.54129
4,48,0.66152,0.01158,650,19,5,2.05611,0.62829


## 2. Elo on its own, for reference

Formula 1 used directly as a forecast, scored on the training years (2014–2022). Every forecast
uses only earlier fights, so this never touches later years. This is the
table from the research page, and it is **not** how K is chosen: on its own,
Elo is best around K ≈ 63, while the model wants something faster.

In [3]:
pd.DataFrame([
    {"K": k, "elo_alone_log_loss": round(elo_forecast_log_loss(fights, k), 5)}
    for k in (16, 31.62, 63.1, 100, 158.5, 199.5, 316.2)
])

,K,elo_alone_log_loss
0,16.00,0.68770
1,31.62,0.68445
2,63.10,0.68151
3,100.00,0.68240
4,158.50,0.69156
5,199.50,0.70302
6,316.20,0.75614


## 3. Choose K by what helps the model

For every K on the grid (10 → 1000 in ~26% steps): attach Elo, take the
training years, score the model by expanding-window CV over 5 seeds. Expect a
broad plateau. Neighbouring K values within ~0.001 of each other are a tie,
not a ranking.

The search judges every K with the settings tuned *without* Elo. Its answer is
therefore checked in section 4, with settings tuned for the model that has Elo.

In [4]:
k_search, curve = choose_elo_k(without_elo, params_without)

no_elo = cv_log_loss(train_without, params_without, seeds=range(5))["log_loss"]
k_table = pd.DataFrame(curve, columns=["K", "cv_log_loss"])
k_table["change_vs_no_elo"] = k_table["cv_log_loss"] - no_elo
print(f"without Elo: {no_elo:.5f}   K chosen by the search: {k_search}")
k_table.round(5)

without Elo: 0.66113   K chosen by the search: 39.81


,K,cv_log_loss,change_vs_no_elo
0,10.00,0.65931,-0.00182
1,12.59,0.65937,-0.00177
2,15.85,0.65885,-0.00228
3,19.95,0.65885,-0.00229
4,25.12,0.65882,-0.00232
5,31.62,0.65935,-0.00179
6,39.81,0.65870,-0.00244
7,50.12,0.66006,-0.00107
8,63.10,0.65954,-0.00159
9,79.43,0.66097,-0.00017


## 4. Check the search's K with the model's own settings

Added on Day 12. The search scores every K with the settings tuned without Elo, and
the right K turned out to depend on those settings: with the 40 inputs, the model
tuned without Elo grows 15-leaf trees, while the model with Elo grows 4-leaf ones.
So the search's K has to beat the previous K, **251.2**, the value every test of the
40 inputs used in the input review (`reviews/end_tests_2026-10-08.md`).

The settings for this check are tuned on the search's K, so the comparison leans
towards the search's K, not the incumbent. The usual rule: adopt the search's K only
if CV is better in at least 7 of 8 seeds **and** men's validation log loss is not worse.

**Result (Day 12):** the search picked K = 39.8. With settings tuned for 39.8, K = 251.2
was still better in CV in all 8 seeds (0.65861 against 0.65939) and on men's validation
(0.6459 against 0.6515), so K stays 251.2 and `elo_params.json` records why. A choice is
only as good as the model judging it: the search's judge was the model without Elo.

In [5]:
PREVIOUS_K = 251.2   # the K every input-review test used
SEEDS = range(8)

def seed_averaged(train_table, valid_table, params):
    """Validation predictions averaged over SEEDS, plus the seed-0 model."""
    X_tr, y_tr = manifest.make_xy(train_table)
    X_va, _ = manifest.make_xy(valid_table)
    models = [fit_model(X_tr, y_tr, params=params | {"random_state": s}) for s in SEEDS]
    return np.mean([predict_proba(m, X_va) for m in models], axis=0), models[0], X_va

tables = {k: chronological_split(build_featured_table(fights, fighters, alphas, elo_k=k))
          for k in dict.fromkeys([k_search, PREVIOUS_K])}
params_search = tune(tables[k_search][0], n_trials=60).best_params   # settings for the search's K

mens = ~is_womens_fight(validation_without, history=without_elo).to_numpy()
y = np.asarray(manifest.make_xy(validation_without)[1], dtype=bool)
check = {}
for k, (train_k, validation_k, _) in tables.items():
    by_seed = cv_log_loss(train_k, params_search, seeds=SEEDS)["log_loss_by_seed"]
    p_k, _, _ = seed_averaged(train_k, validation_k, params_search)
    check[k] = {"by_seed": np.array(by_seed), "validation": score(y[mens], p_k[mens])["log_loss"]}

change = check[k_search]["by_seed"] - check[PREVIOUS_K]["by_seed"]
search_wins = (change < 0).sum() >= 7 and check[k_search]["validation"] <= check[PREVIOUS_K]["validation"]
best_k = k_search if search_wins else PREVIOUS_K
note = None if search_wins else (
    f"the search's K = {k_search} lost the check against K = {PREVIOUS_K} with settings tuned "
    f"for the search's K: CV better in {(change < 0).sum()} of 8 seeds, men's validation log loss "
    f"{check[k_search]['validation']:.4f} vs {check[PREVIOUS_K]['validation']:.4f} (05_elo.ipynb, section 4)")
save_elo_params(best_k, curve, seeds=range(5), note=note)

print(f"search's K better in {(change < 0).sum()} of 8 seeds (mean change {change.mean():+.5f})")
print(f"chosen K: {best_k}" + ("" if search_wins else f"  (the search's {k_search} lost the check)"))
pd.DataFrame({k: {"cv_log_loss": v["by_seed"].mean(), "men_validation_log_loss": v["validation"]}
              for k, v in check.items()}).T.rename_axis("K").round(5)

search's K better in 0 of 8 seeds (mean change +0.00077)
chosen K: 251.2  (the search's 39.81 lost the check)


,cv_log_loss,men_validation_log_loss
K,,
39.81,0.65939,0.65147
251.20,0.65861,0.64591


## 5. Rebuild the featured table with Elo

The same shared chain as notebook 03, now with the K from section 4. Spot-check: UFC 69,
where Matt Serra upset Georges St-Pierre.

In [6]:
featured = build_featured_table(fights, fighters, alphas, elo_k=best_k)
save_processed(featured, "fights_featured")

assert manifest.unclassified_columns(featured) == []
print(f"{featured.shape[1]} columns, {len(manifest.feature_columns(featured))} features")
featured.loc[
    featured["event_date"].eq("2007-04-07") & featured["fighter_a_name"].eq("Matt Serra"),
    ["fighter_a_name", "fighter_b_name", *ELO_COLUMNS, "diff_prior_elo"],
].round(1)

373 columns, 40 features


,fighter_a_name,fighter_b_name,fighter_a_prior_elo,fighter_b_prior_elo,diff_prior_elo
713,Matt Serra,Georges St-Pierre,1641.0,2241.9,-600.9


## 6. Re-tune with Elo

The feature set changed, so the settings are re-chosen exactly the same way.
This saves `model_params.json` again, now for the table with Elo.
If section 4 kept the search's K, this repeats its tuning exactly (same table, same seed).

In [7]:
train, validation, _ = chronological_split(featured)      # the test years stay closed
study_with = tune(train, n_trials=60)
save_params(study_with)
params_with = study_with.best_params
print(f"best CV log loss, averaged over 3 seeds: {study_with.best_value:.5f}")
tuning_table(study_with, top=5)

best CV log loss, averaged over 3 seeds: 0.65752


,trial,log_loss,learning_rate,max_iter,max_leaf_nodes,min_samples_leaf,l2_regularization,max_features
0,56,0.65752,0.02126,700,5,81,24.71659,0.47733
1,23,0.65763,0.01405,900,5,79,4.28156,0.57350
2,51,0.65782,0.01809,900,6,68,29.88596,0.47176
3,58,0.65804,0.01665,850,6,75,14.44067,0.50310
4,33,0.65808,0.01916,800,4,66,22.62280,0.56510


## 7. The decision

Each version with its own tuned settings. First cross-validation over 8 seeds,
compared seed by seed. Then the one look at validation, with predictions
averaged over the same 8 seeds. The rule set before running: keep Elo if CV
improves in at least 7 of 8 seeds and validation doesn't contradict it.

In [8]:
cv_without = cv_log_loss(train_without, params_without, seeds=SEEDS)
cv_with = cv_log_loss(train, params_with, seeds=SEEDS)
change = np.array(cv_with["log_loss_by_seed"]) - np.array(cv_without["log_loss_by_seed"])
print(f"CV log loss: without Elo {cv_without['log_loss']:.5f} -> with Elo {cv_with['log_loss']:.5f}")
print(f"change {change.mean():+.5f}, better in {(change < 0).sum()} of 8 seeds")

CV log loss: without Elo 0.66122 -> with Elo 0.65760
change -0.00363, better in 8 of 8 seeds


In [9]:
assert (validation_without["fights_url"].to_numpy() == validation["fights_url"].to_numpy()).all()

p_without, _, _ = seed_averaged(train_without, validation_without, params_without)
p_with, model_with, X_val = seed_averaged(train, validation, params_with)
_, y_val = manifest.make_xy(validation)

result = compare(y[mens], p_with[mens], p_without[mens])
print(f"men's fights, with vs without Elo: fixes {result['fixes']}, breaks {result['breaks']}, "
      f"p = {result['p_value']:.3f}")
scorecard({"without Elo": p_without, "with Elo": p_with}, y_val, mens)

men's fights, with vs without Elo: fixes 73, breaks 57, p = 0.188


,men_accuracy,men_log_loss,men_auc,all_accuracy,all_log_loss,all_auc
model,,,,,,
without Elo,0.5938,0.6551,0.6507,0.5939,0.6591,0.6430
with Elo,0.6129,0.6455,0.6749,0.6136,0.6481,0.6694


In [10]:
# which feature families the tuned model with Elo relies on (men's validation fights)
groups = manifest.feature_groups(train)
group_importance(model_with, X_val[mens], y_val[mens], groups, repeats=20)

,group,columns,log_loss_rise,spread
0,physical,5,0.028949,0.004310
1,strength,1,0.018700,0.004999
2,defence,11,0.014853,0.003599
3,striking,13,0.008971,0.003264
4,grappling,4,0.007516,0.003308
5,cold_start,2,0.000861,0.000803
6,damage,1,-0.000002,0.000608
7,record,3,-0.000959,0.000788
